# Margin, collateral, and XVA

**Prerequisites:** Work through `01_foundations/core_types_and_money.ipynb`, `01_foundations/dates_calendars_schedules.ipynb`, `01_foundations/market_data_and_curves.ipynb`, and `01_foundations/math_toolkit.ipynb`. Those notebooks introduce currencies, dates, and basic finstack patterns used implicitly here.

This notebook introduces the `finstack_quant.margin` API: netting sets, CSA specs, variation margin (VM), eligible collateral schedules, utilization metrics, and configuration types that feed XVA-style workflows.

**Scope note:** This notebook focuses on **margin and collateral configuration** (netting sets, CSA, VM, eligible collateral, utilization, and `FundingConfig` / `XvaConfig` as configuration types). A full **CVA** walkthrough (exposure profiles, netting, and discounting) is intentionally out of scope here and could be added later as a separate pedagogical section.

## Concepts: margin, collateral, and XVA

**Variation margin (VM)** is the bilateral exchange of collateral driven by mark-to-market exposure under a **Credit Support Annex (CSA)**. Thresholds, independent amounts, and minimum transfer amounts determine when a **margin call** is required and how much must be delivered or returned.

**Collateral eligibility** (cash vs. securities, haircuts, rehypothecation) is captured in schedules such as BCBS-style baskets or cash-only setups.

**XVA** (collectively CVA, DVA, FVA, etc.) adjusts derivative values for credit, funding, and related effects. Full XVA engines simulate exposure paths over time; here we focus on **configuration types** — for example `FundingConfig` encodes funding spread and benefit assumptions used when pricing funding costs (FVA) or related adjustments.

The API below is **self-contained**: identifiers, CSA presets, VM calculation, collateral schedules, utilization, and XVA/funding configuration.

## API walkthrough: imports and availability

We import the main margin types from `finstack_quant.margin`. If the extension module is missing (e.g. wheel not built), we record the error so later cells can skip gracefully while still printing something.

In [1]:
from finstack_quant.margin import (
    NettingSetId,
    CsaSpec,
    VmCalculator,
    XvaConfig,
    FundingConfig,
    MarginUtilization,
    EligibleCollateralSchedule,
)


print("finstack_quant.margin: imports OK")


finstack_quant.margin: imports OK


### Netting set identifiers

A **netting set** groups trades for which close-out and margin apply together. **Bilateral** sets are keyed by counterparty and CSA id; **cleared** sets are keyed by CCP id.

In [2]:
ns_bilateral = NettingSetId.bilateral("CPTY-1", "CSA-001")
ns_cleared = NettingSetId.cleared("LCH")
print(f"Bilateral: {ns_bilateral}")
print(f"Cleared: {ns_cleared}")


Bilateral: CPTY-1:CSA-001
Cleared: CCP:LCH


### `CsaSpec`: regulatory-style CSA presets

`CsaSpec` wraps the legal/economic terms needed for VM logic. Factory methods such as `usd_regulatory()` and `eur_regulatory()` provide ready-made specs; `to_json()` supports serialization.

In [3]:
csa = CsaSpec.usd_regulatory()
print(f"CSA: {csa}")
print(f"JSON: {csa.to_json()}")
csa_eur = CsaSpec.eur_regulatory()
print(f"EUR CSA: {csa_eur}")


CSA: CsaSpec(id="USD-REGULATORY-CSA", currency=USD, requires_im=true)
JSON: {
  "id": "USD-REGULATORY-CSA",
  "base_currency": "USD",
  "calendar_id": "usny",
  "vm_params": {
    "threshold": {
      "amount": "0",
      "currency": "USD"
    },
    "mta": {
      "amount": "500000",
      "currency": "USD"
    },
    "rounding": {
      "amount": "10000",
      "currency": "USD"
    },
    "independent_amount": {
      "amount": "0",
      "currency": "USD"
    },
    "frequency": "Daily",
    "settlement_lag": 1
  },
  "im_params": {
    "methodology": "Simm",
    "mpor_days": 10,
    "threshold": {
      "amount": "50000000",
      "currency": "USD"
    },
    "mta": {
      "amount": "0",
      "currency": "USD"
    },
    "segregated": true
  },
  "eligible_collateral": {
    "eligible": [
      {
        "asset_class": "cash",
        "haircut": 0.0,
        "fx_haircut_addon": 0.08
      },
      {
        "asset_class": "government_bonds",
        "min_rating": "AA-",
        

### `VmCalculator`: variation margin from exposure and posted collateral

Given mark-to-market **exposure**, **posted collateral**, **currency**, and a **valuation date** (`year`, `month`, `day`), the calculator returns gross/net exposure, delivery and return amounts, net margin, and whether a **margin call** is required.

In [4]:
vm = VmCalculator(CsaSpec.usd_regulatory())
result = vm.calculate(1_000_000.0, 0.0, "USD", 2025, 1, 15)
print(f"Gross exposure: {result.gross_exposure:,.2f}")
print(f"Net exposure: {result.net_exposure:,.2f}")
print(f"Delivery amount: {result.delivery_amount:,.2f}")
print(f"Return amount: {result.return_amount:,.2f}")
print(f"Net margin: {result.net_margin:,.2f}")
print(f"Requires call: {result.requires_call}")
result2 = vm.calculate(-500_000.0, 200_000.0, "USD", 2025, 1, 15)
print(
    f"\nNeg exposure: gross={result2.gross_exposure:,.2f}, net={result2.net_exposure:,.2f}"
)
print(f"Requires call: {result2.requires_call}")


Gross exposure: 1,000,000.00
Net exposure: 1,000,000.00
Delivery amount: 1,000,000.00
Return amount: 0.00
Net margin: 1,000,000.00
Requires call: True

Neg exposure: gross=-500,000.00, net=-500,000.00
Requires call: True


### `EligibleCollateralSchedule`: what can be posted

Schedules describe which asset classes are eligible and with what haircuts — from **cash-only** to **BCBS-standard** baskets or **US Treasury** schedules.

In [5]:
cash = EligibleCollateralSchedule.cash_only()
print(f"Cash only: {cash}")
bcbs = EligibleCollateralSchedule.bcbs_standard()
print(f"BCBS standard: {bcbs}")
ust = EligibleCollateralSchedule.us_treasuries()
print(f"US Treasuries: {ust}")


Cash only: EligibleCollateralSchedule(eligible=1, rehyp=false)
BCBS standard: EligibleCollateralSchedule(eligible=4, rehyp=false)
US Treasuries: EligibleCollateralSchedule(eligible=4, rehyp=true)


### `MarginUtilization`: posted vs. required

`MarginUtilization(posted_amount, required_amount, currency)` summarizes how fully margin covers the requirement (ratio, adequacy, shortfall).

In [6]:
util = MarginUtilization(1_200_000.0, 1_000_000.0, "USD")
print(f"Utilization: {util}")
print(f"Posted: {util.posted:,.2f}, required: {util.required:,.2f}, ratio: {util.ratio:.4f}")
print(f"Adequate: {util.is_adequate()}, shortfall: {util.shortfall():,.2f}")


Utilization: MarginUtilization(ratio=1.20, adequate=true)
Posted: 1,200,000.00, required: 1,000,000.00, ratio: 1.2000
Adequate: True, shortfall: 0.00


### `FundingConfig` and `XvaConfig`

`FundingConfig(funding_spread_bps, funding_benefit_bps=None)` holds spreads used in funding-aware valuation. `XvaConfig()` builds a default time grid and recovery assumptions; it can optionally embed a `FundingConfig`.

In [7]:
funding = FundingConfig(50.0, 30.0)
print(f"Funding: {funding}")
print(
    f"spread_bps={funding.funding_spread_bps}, "
    f"benefit_bps={funding.funding_benefit_bps}, "
    f"effective_benefit_bps={funding.effective_benefit_bps()}"
)
xva_cfg = XvaConfig()
print(f"XVA config: {xva_cfg}")
print(f"Time grid length: {len(xva_cfg.time_grid)}, recovery_rate: {xva_cfg.recovery_rate}")


Funding: FundingConfig(spread=50.0bp, benefit=Some(30.0)bp)
spread_bps=50.0, benefit_bps=30.0, effective_benefit_bps=30.0
XVA config: XvaConfig(grid_points=120, recovery=40%)
Time grid length: 120, recovery_rate: 0.4


## Mini-example: bilateral CSA, VM, and funding cost

1. Use a **USD regulatory** CSA (typical bilateral VM context).
2. Portfolio **+$5M** exposure with **$1M** already posted.
3. Inspect **margin call** flags and amounts.
4. Flip to **negative exposure** and see how VM logic responds.
5. Price the **funding cost** of posted margin with `MarginFundingCost`, which turns a funding rate, a collateral remuneration rate, and the posted amount into a spread, an annual cost, and a cost for any accrual period.

In [8]:
ns = NettingSetId.bilateral("ACME-DEALER", "CSA-USD-REG")
csa_usd = CsaSpec.usd_regulatory()
calc = VmCalculator(csa_usd)
exposure_pos = 5_000_000.0
posted = 1_000_000.0
as_of = (2025, 4, 11)
r = calc.calculate(exposure_pos, posted, "USD", *as_of)
print(f"Netting set: {ns}")
print(f"CSA: {csa_usd}")
print(f"Exposure +$ {exposure_pos:,.0f}, posted $ {posted:,.0f}")
print(f"Gross exposure: {r.gross_exposure:,.2f}")
print(f"Net exposure: {r.net_exposure:,.2f}")
print(f"Delivery: {r.delivery_amount:,.2f}, return: {r.return_amount:,.2f}")
print(f"Net margin: {r.net_margin:,.2f}")
print(f"Requires margin call: {r.requires_call}")
u = MarginUtilization(posted, exposure_pos, "USD")
print(f"Posted vs gross MTM (illustrative): {u}")
print(f"  ratio={u.ratio:.4f}, adequate={u.is_adequate()}, shortfall={u.shortfall():,.2f}")


Netting set: ACME-DEALER:CSA-USD-REG
CSA: CsaSpec(id="USD-REGULATORY-CSA", currency=USD, requires_im=true)
Exposure +$ 5,000,000, posted $ 1,000,000
Gross exposure: 5,000,000.00
Net exposure: 5,000,000.00
Delivery: 4,000,000.00, return: 0.00
Net margin: 4,000,000.00
Requires margin call: True
Posted vs gross MTM (illustrative): MarginUtilization(ratio=0.20, adequate=false)
  ratio=0.2000, adequate=False, shortfall=4,000,000.00


In [9]:
calc = VmCalculator(CsaSpec.usd_regulatory())
exposure_neg = -5_000_000.0
posted_still = 1_000_000.0
r_rev = calc.calculate(exposure_neg, posted_still, "USD", 2025, 4, 11)
print("Exposure reversed (we owe / negative MTM):")
print(f"Gross exposure: {r_rev.gross_exposure:,.2f}")
print(f"Net exposure: {r_rev.net_exposure:,.2f}")
print(f"Delivery: {r_rev.delivery_amount:,.2f}, return: {r_rev.return_amount:,.2f}")
print(f"Net margin: {r_rev.net_margin:,.2f}")
print(f"Requires margin call: {r_rev.requires_call}")


Exposure reversed (we owe / negative MTM):
Gross exposure: -5,000,000.00
Net exposure: -5,000,000.00
Delivery: 6,000,000.00, return: 0.00
Net margin: 6,000,000.00
Requires margin call: True


In [10]:
from finstack_quant.margin import MarginFundingCost

funding = FundingConfig(50.0, 30.0)
margin_posted = 4_000_000.0

# `MarginFundingCost` owns the funding arithmetic: it takes the rate we fund at and
# the rate the posted collateral earns, and derives spread, annual cost, and the
# cost over any accrual period. Here collateral is remunerated at the overnight
# rate and we fund at that rate plus the CSA funding spread.
collateral_rate = 0.0430
funding_rate = collateral_rate + funding.funding_spread_bps / 10_000.0
cost = MarginFundingCost(margin_posted, funding_rate, collateral_rate, "USD")

print(
    f"Funding ${margin_posted:,.0f} of posted margin at {funding_rate:.4%} "
    f"against collateral earning {collateral_rate:.4%}"
)
print(f"  spread:      {cost.spread():.4%} ({funding.funding_spread_bps:.0f} bp)")
print(f"  annual cost: ${cost.annual_cost:,.2f}")
print(f"  90-day cost: ${cost.cost_for_period(0.25):,.2f}")
print(f"FundingConfig repr: {funding}")
xva_with_funding = XvaConfig(funding=funding)
print(f"XvaConfig with embedded funding: {xva_with_funding}")

Funding $4,000,000 of posted margin at 4.8000% against collateral earning 4.3000%
  spread:      0.5000% (50 bp)
  annual cost: $20,000.00
  90-day cost: $5,000.00
FundingConfig repr: FundingConfig(spread=50.0bp, benefit=Some(30.0)bp)
XvaConfig with embedded funding: XvaConfig(grid_points=120, recovery=40%)


## Result & value types across the margin surface

The margin module exposes typed results (`VmResult`, `ExposureProfile`, `XvaResult`), collateral/funding value types (`CsaTerms`, `XvaNettingSet`, `ExcessCollateral`, `MarginFundingCost`, `Haircut01`), and lifecycle enums (`ClearingStatus`, `MarginCallType`, `MarginTenor`), plus module `CONSTANTS`.

In [11]:
import json

import finstack_quant.margin as m

# CSA terms and an XVA netting set built from them
terms = m.CsaTerms(threshold=0.0, mta=250_000.0, mpor_days=10, independent_amount=0.0)
nset = m.XvaNettingSet("NS-1", "CPTY-A", terms)
print("CsaTerms.mpor_days =", terms.mpor_days, "| XvaNettingSet collateralized =", nset.is_collateralized)

# Collateral value types
print("ExcessCollateral.excess =", m.ExcessCollateral(1_200_000.0, 1_000_000.0, "USD").excess)
print("Haircut01.pv_change =", m.Haircut01(1_000_000.0, 2.0, "USD").pv_change)

# Lifecycle enums
print("ClearingStatus:", m.ClearingStatus.bilateral(), "/", m.ClearingStatus.cleared("LCH"))
print("MarginCallType:", m.MarginCallType.initial_margin(), "/", m.MarginCallType.variation_margin_delivery())
print("MarginTenor:", m.MarginTenor.daily(), m.MarginTenor.weekly(), m.MarginTenor.monthly(), m.MarginTenor.on_demand())

# Exposure profile and XVA result round-trips (JSON-first deep results)
exposure = m.ExposureProfile.from_json(json.dumps({
    "times": [0.0, 0.5, 1.0], "mtm_values": [0.0, 50.0, 40.0],
    "epe": [0.0, 100.0, 80.0], "ene": [0.0, 20.0, 10.0],
}))
print("ExposureProfile EPE:", exposure.epe)
xva = m.XvaResult.from_json(json.dumps({
    "cva": 1000.0, "dva": 200.0, "fva": 50.0, "bilateral_cva": 800.0,
    "epe_profile": [[0.0, 0.0], [0.5, 100.0], [1.0, 80.0]],
    "ene_profile": [[0.0, 0.0], [0.5, 20.0], [1.0, 10.0]],
    "pfe_profile": [[0.0, 0.0], [0.5, 100.0], [1.0, 80.0]],
    "max_pfe": 100.0,
    "effective_epe_profile": [[0.0, 0.0], [0.5, 100.0], [1.0, 100.0]],
    "effective_epe": 75.0,
}))
print("XvaResult: cva =", xva.cva, "| dva =", xva.dva, "| max_pfe =", xva.max_pfe)

# Module-level regulatory constants
print("CONSTANTS keys (sample):", list(m.CONSTANTS)[:5])

CsaTerms.mpor_days = 10 | XvaNettingSet collateralized = True
ExcessCollateral.excess = 200000.0
Haircut01.pv_change = 100.0
ClearingStatus: bilateral / cleared:LCH
MarginCallType: MarginCallType(InitialMargin) / MarginCallType(VariationMarginDelivery)
MarginTenor: daily weekly monthly on_demand
ExposureProfile EPE: [0.0, 100.0, 80.0]
XvaResult: cva = 1000.0 | dva = 200.0 | max_pfe = 100.0
CONSTANTS keys (sample): ['BCBS_IOSCO_SCHEDULE_ID']


## Takeaways

- **NettingSetId** distinguishes bilateral (counterparty + CSA) from cleared (CCP) contexts; use consistent ids across risk and margin systems.
- **CsaSpec** drives **VmCalculator**: thresholds and CSA terms determine **delivery**, **return**, **net margin**, and **requires_call**.
- **EligibleCollateralSchedule** presets model what can be posted (cash, BCBS-style, Treasuries) for policy and haircut logic.
- **MarginUtilization** summarizes **posted vs. required** with ratio, adequacy, and shortfall.
- **FundingConfig** / **XvaConfig** connect margin and funding assumptions to XVA-style configuration, and **MarginFundingCost** turns the funding-vs-collateral rate differential into an annual (or per-period) cost of carrying posted margin.

Next steps: combine this with pricing notebooks (e.g. **05–06**) to feed exposures into margin and, where available, full XVA engines.